## Proyecto punta a punta

Cuántos terremotos habrá mañana en Tenerife


#### Forecasting:

## DATOS :

Un técnico en vulcanología quiere saber cuantos terremotos podrían generarse en el día de mañana en Tenerife para saber si es normal o es raro la cantidad.

Datos -> catálogo adquirido del IGN de los terremotos mayores a 1,0. En csv.

* 2 archivos -> desde 1981 hasta 2017, y de 2017 hasta hoy.

No predice erupciones que iba a ser el tema principal. La última fue Chinyero 1909. Solo predice el número de terremotos y veremos si la magnitud del mismo.

In [1]:
! pip install pandas numpy matplotlib seaborn joblib scikit-learn

  Using cached pandas-3.0.6-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.11.2-cp314-cp314-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached contourpy-1.4.0-cp314-cp314-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.66.1-cp314-cp314-win_amd64.whl.metadata (132 kB)
  Using cached kiwisolver-1.5.1-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [12]:
# Los archivos del IGN vienen separados por ; y con espacios
# Después de descubrir por qué no se cargaban bien los datos, descubrí el qué era y lo busqué para solucionarlo. El problema era que los archivos del IGN vienen separados por ; y con espacios, lo que hacía que al leerlos con pandas, se generaran columnas vacías y no se cargaran correctamente los datos. 
# Para solucionarlo, se puede usar el parámetro sep=';' y skipinitialspace=True al leer el archivo con pandas.

df1 = pd.read_csv("../data/catalogo_ign_1981_2017.csv", sep=";", skipinitialspace=True)
df2 = pd.read_csv("../data/catalogo_ign_2017_2026.csv", sep=";", skipinitialspace=True)

print('df1:', df1.shape)
print('df2:', df2.shape)

df1.head()

df1: (25740, 10)
df2: (94004, 10)


,Evento,Fecha,Hora,Latitud,Longitud,Prof. (Km),Inten.,Mag.,Tipo Mag.,Localizaci�n
0,9044,22/04/1981,17:41:12,28.099,-16.174,5.0,V,3.8,13.0,"E San Miguel de Tajao.TF,Zona atlántica Teneri..."
1,9183,13/10/1981,05:28:24,28.013,-17.862,0.0,NaN,2.4,13.0,"N Isla de El Hierro,Oceáno Atlántico"
2,es1981vxaza,08/11/1981,10:55:27,29.122,-16.271,0.0,NaN,2.8,13.0,"N Isla de Tenerife,Oceáno Atlántico"
3,9202,16/11/1981,02:31:28,29.987,-15.375,5.0,NaN,3.8,13.0,"NE Isla de Tenerife,Oceáno Atlántico"
4,9322,25/04/1982,04:15:32,28.826,-16.477,5.0,NaN,2.7,13.0,"N Isla de Tenerife,Oceáno Atlántico"


In [ ]:
# Juntamos las 2 tablas en 1 sola

df = pd.concat([df1, df2], ignore_index=True)

# Los nombres de la columna traen espacios y algunos con caracteres raros así que solucionamos

df.columns = ["evento","fecha","hora","latitud","longitud","profundidad","intensidad","magnitud","tipo_mag","localizacion"]

filas_inicio = len(df)
print("Filas al inicio:", filas_inicio)
df.tail() # mostramos las últimas filas para ver si hay algo raro en los datos

Filas al inicio: 119744


,evento,fecha,hora,latitud,longitud,profundidad,intensidad,magnitud,tipo_mag,localizacion
119739,es2026toxro,06/10/2026,11:38:03,37.0712,-3.5997,4.0,NaN,1.8,4.0,SE DÍLAR.GR
119740,es2026tpabg,06/10/2026,12:48:43,42.9675,0.1724,0.0,NaN,2.1,4.0,NW LA MONGIE.FRA
119741,es2026tpbfq,06/10/2026,13:24:07,37.0123,-3.5731,4.0,NaN,1.6,4.0,NW DÚRCAL.GR
119742,es2026tpbqu,06/10/2026,13:37:07,37.0894,-3.5525,6.0,NaN,1.3,4.0,SE LA ZUBIA.GR
119743,es2026tpbqy,06/10/2026,13:37:17,37.1058,-3.5717,6.0,NaN,1.5,4.0,SE LA ZUBIA.GR


#### Las columnas que me interesan son: Fecha, longitud, latitud y magnitud.

## Revisamos nulos/duplicados

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119744 entries, 0 to 119743
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   evento        119744 non-null  str    
 1   fecha         119744 non-null  str    
 2   hora          119744 non-null  str    
 3   latitud       119744 non-null  float64
 4   longitud      119744 non-null  float64
 5   profundidad   119744 non-null  float64
 6   intensidad    7181 non-null    str    
 7   magnitud      119741 non-null  float64
 8   tipo_mag      119741 non-null  float64
 9   localizacion  119744 non-null  str    
dtypes: float64(5), str(5)
memory usage: 9.1 MB


In [15]:
# nulos

df.isnull().sum()

evento               0
fecha                0
hora                 0
latitud              0
longitud             0
profundidad          0
intensidad      112563
magnitud             3
tipo_mag             3
localizacion         0
dtype: int64

In [16]:
df = df.drop(columns=["intensidad", "tipo_mag"]) #eliminamos las columnas que no nos interesan
df = df.dropna(subset=["magnitud"]) #eliminamos las filas con valores nulos

filas_sin_nulos = len(df)
print("Sin nulos:", filas_sin_nulos)

Sin nulos: 119741


#### Vemos que pueden haber duplicados en los datos, ya que ambos archivos tienen el año 2017.

#### Procedemos a revisar y a eliminarlos

In [17]:
df["evento"] = df["evento"].astype(str).str.strip() # eliminamos los espacios en blanco al inicio y al final de la columna evento
print("Repetidos: ", df.duplicated(subset="evento").sum()) # contamos los duplicados en la columna evento

Repetidos:  720


#### Efectivamente hay 720 eventos repetidos

Los quitamos

In [18]:
df = df.drop_duplicates(subset="evento") # eliminamos los duplicados en la columna evento
filas_sin_duplicados = len(df)
print("Sin duplicados:", filas_sin_duplicados)


Sin duplicados: 119021


#### Cambiamos el formato de la fecha de texto a datetime

In [19]:
df["fecha"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y") #convertimos la columna fecha a tipo datetime
print("Primera fecha:", df["fecha"].min())
print("Última fecha:", df["fecha"].max())

Primera fecha: 1981-04-22 00:00:00
Última fecha: 2026-10-06 00:00:00
